<a href="https://colab.research.google.com/github/vahnico/pragmatic-analyzer/blob/main/parsingwajahtubuqwen.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==============================================================
# CELL 1: KONFIGURASI PIPELINE
# Strategi: Qwen2.5-VL 7B + single structured prompt + batch inference
# Target: 100 video panjang, efisiensi maksimal di L4
# Author: Dr. Hendi Pratama, S.Pd., M.A.
# ==============================================================

# === PARAMETER UTAMA ===
SAMPLING_INTERVAL = 5        # detik antara frame (3 untuk granular, 5 untuk efisien)
TRANSCRIPT_WINDOW = 5        # detik ke belakang untuk context transkrip

# === MODEL ===
WHISPER_MODEL = "turbo"      # Whisper Turbo: cepat dan akurat
VLM_MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"   # 7B: speed + quality sweet spot

# === VLM INFERENCE PARAMETER ===
VLM_BATCH_SIZE = 4           # jumlah frame per batch (L4 bisa handle 4-6)
VLM_MAX_NEW_TOKENS = 350     # cukup untuk structured output 3 section

# === PATH ===
DRIVE_VIDEO_FOLDER = "/content/drive/MyDrive/video_downloads"
DRIVE_OUTPUT_FOLDER = "/content/drive/MyDrive/multimodal_analysis_output"

# === VERIFIKASI GPU ===
import subprocess
gpu_info = subprocess.check_output("nvidia-smi", shell=True).decode()
print(gpu_info)

print("=" * 60)
print("KONFIGURASI PIPELINE")
print("=" * 60)
print(f"Interval sampling      : {SAMPLING_INTERVAL} detik")
print(f"Window transkrip       : {TRANSCRIPT_WINDOW} detik ke belakang")
print(f"Model Whisper          : {WHISPER_MODEL}")
print(f"Model VLM              : {VLM_MODEL_ID}")
print(f"Batch size VLM         : {VLM_BATCH_SIZE} frame per inferensi")
print(f"Max tokens VLM         : {VLM_MAX_NEW_TOKENS}")
print(f"Folder video input     : {DRIVE_VIDEO_FOLDER}")
print(f"Folder output analisis : {DRIVE_OUTPUT_FOLDER}")

Sat Apr 18 04:25:23 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   43C    P8             13W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# ==============================================================
# CELL 2: INSTALASI DEPENDENCIES
# Versi verbose (tanpa -q) agar error langsung terlihat
# ==============================================================

# System dependencies
!apt-get install -y ffmpeg -qq

# Python packages
!pip install openai-whisper
!pip install "transformers>=4.45.0" "accelerate>=0.34.0"
!pip install qwen-vl-utils
!pip install openpyxl

print("\n" + "=" * 60)
print("VERIFIKASI INSTALASI")
print("=" * 60)

import importlib
packages = {
    "whisper": "openai-whisper",
    "transformers": "transformers",
    "accelerate": "accelerate",
    "qwen_vl_utils": "qwen-vl-utils",
    "openpyxl": "openpyxl",
    "torch": "torch",
    "PIL": "pillow",
    "pandas": "pandas",
    "numpy": "numpy"
}

all_ok = True
for mod, pkg in packages.items():
    try:
        m = importlib.import_module(mod)
        v = getattr(m, "__version__", "?")
        print(f"  [OK]   {pkg:20s} {v}")
    except ImportError as e:
        print(f"  [FAIL] {pkg:20s} {str(e)[:60]}")
        all_ok = False

if all_ok:
    print("\nSemua package siap. Lanjut ke Cell 3.")
else:
    print("\nAda package gagal. Laporkan ke asisten.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 44.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for openai-whisper: filename=openai_whisper-20250625-py3-none-any.whl size=803979 sha256=46b4eca8816c019b28f67f78a5676e0dec010a80bc3b2e7ec581e09252d74a0f
  Stored in directory: /root/.cache/pip/wheels/61/d2/20/09ec9bef734d126cba375b15898010b6cc28578d8afdde5869
Successfully built openai-whisper
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.4/36.4 MB 73.2 MB/s eta 0:00:00

VERIFIKASI INSTALASI
  [OK]   openai-whisper       20250625
  [OK]   transformers         5.0.0
  [OK]   accelerate           1.13.0
  [OK]   qwen-vl-utils        ?
  [OK]   openpyxl             3.1.5
  [OK]   torch                2.10.0+cu128
  [OK]   pillow               11.3.0
  [OK]   pandas               2.2.2
  [OK]   numpy                2.0.2

Semua package siap. Lanjut ke Cell 3.


In [ ]:
# ==============================================================
# CELL 3: DIALOG PEMILIHAN VIDEO DARI GOOGLE DRIVE
# Support: single video atau batch multi-video
# ==============================================================

from google.colab import drive
from pathlib import Path
import os, re
import ipywidgets as widgets
from IPython.display import display, clear_output

# Mount Drive
if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')

# Pastikan folder input dan output ada
video_folder_path = Path(DRIVE_VIDEO_FOLDER)
video_folder_path.mkdir(parents=True, exist_ok=True)
Path(DRIVE_OUTPUT_FOLDER).mkdir(parents=True, exist_ok=True)

# Scan semua file video
SUPPORTED_EXTS = ['.mp4', '.mov', '.avi', '.mkv', '.webm', '.flv', '.m4v']
video_files_in_drive = []
for ext in SUPPORTED_EXTS:
    video_files_in_drive.extend(sorted(video_folder_path.glob(f"*{ext}")))
    video_files_in_drive.extend(sorted(video_folder_path.glob(f"*{ext.upper()}")))
video_files_in_drive = sorted(set(video_files_in_drive))

if not video_files_in_drive:
    raise FileNotFoundError(
        f"Tidak ada video di {DRIVE_VIDEO_FOLDER}.\n"
        f"Upload video via drive.google.com, lalu ulang cell ini."
    )

print(f"Ditemukan {len(video_files_in_drive)} video di Drive:\n")
for i, f in enumerate(video_files_in_drive, 1):
    size_mb = f.stat().st_size / (1024 * 1024)
    print(f"  {i}. {f.name} ({size_mb:.2f} MB)")

# Widget: mode pilih (single atau batch)
mode_widget = widgets.RadioButtons(
    options=[
        ('Single video (pilih satu dari dropdown)', 'single'),
        ('Batch multi-video (pilih beberapa sekaligus)', 'batch'),
        ('Batch SEMUA video di folder', 'all')
    ],
    value='single',
    description='Mode:',
    layout=widgets.Layout(width='90%'),
    style={'description_width': 'initial'}
)

# Widget: single dropdown
single_dropdown = widgets.Dropdown(
    options=[(f.name, str(f)) for f in video_files_in_drive],
    description='Pilih video:',
    layout=widgets.Layout(width='90%'),
    style={'description_width': 'initial'}
)

# Widget: multi-select untuk batch
multi_select = widgets.SelectMultiple(
    options=[(f.name, str(f)) for f in video_files_in_drive],
    description='Pilih video (Ctrl/Shift-click):',
    layout=widgets.Layout(width='90%', height='200px'),
    style={'description_width': 'initial'}
)

confirm_btn = widgets.Button(
    description='Konfirmasi Pilihan',
    button_style='primary',
    layout=widgets.Layout(width='250px')
)

output_area = widgets.Output()

# State global
selected_videos = {"paths": [], "mode": None}

print("\nPilih mode dan video:\n")
display(mode_widget, single_dropdown, multi_select, confirm_btn, output_area)


def on_confirm(btn):
    with output_area:
        clear_output()
        mode = mode_widget.value
        selected_videos["mode"] = mode

        if mode == 'single':
            selected_videos["paths"] = [single_dropdown.value]
        elif mode == 'batch':
            if not multi_select.value:
                print("Belum pilih video di multi-select. Silakan pilih dulu.")
                return
            selected_videos["paths"] = list(multi_select.value)
        elif mode == 'all':
            selected_videos["paths"] = [str(f) for f in video_files_in_drive]

        print(f"Mode terpilih: {mode}")
        print(f"Jumlah video : {len(selected_videos['paths'])}\n")
        print("Daftar video yang akan diproses:")
        total_size = 0
        for i, p in enumerate(selected_videos["paths"], 1):
            path_obj = Path(p)
            size_mb = path_obj.stat().st_size / (1024 * 1024)
            total_size += size_mb
            print(f"  {i}. {path_obj.name} ({size_mb:.2f} MB)")
        print(f"\nTotal ukuran: {total_size:.2f} MB")
        print("\nLanjut ke Cell 4.")


confirm_btn.on_click(on_confirm)

Mounted at /content/drive
Ditemukan 1 video di Drive:

  1. Ini kata pendiri Zenius ＂Sabda PS＂ [PRLQSBl1v-I].mp4 (2.40 MB)

Pilih mode dan video:



RadioButtons(description='Mode:', layout=Layout(width='90%'), options=(('Single video (pilih satu dari dropdow…

Dropdown(description='Pilih video:', layout=Layout(width='90%'), options=(('Ini kata pendiri Zenius ＂Sabda PS＂…

SelectMultiple(description='Pilih video (Ctrl/Shift-click):', layout=Layout(height='200px', width='90%'), opti…

Button(button_style='primary', description='Konfirmasi Pilihan', layout=Layout(width='250px'), style=ButtonSty…

Output()

In [ ]:
# ==============================================================
# CELL 4: EKSTRAKSI FRAME - DIBUNGKUS DALAM FUNGSI REUSABLE
# Fungsi ini dipanggil per video di Cell 10 (batch loop)
# ==============================================================

import subprocess, json, shutil
from pathlib import Path
from tqdm import tqdm


def get_video_duration(video_path):
    """Ambil durasi video dalam detik via ffprobe."""
    probe = subprocess.run(
        ["ffprobe", "-v", "error", "-show_entries", "format=duration",
         "-of", "json", str(video_path)],
        capture_output=True, text=True
    )
    return float(json.loads(probe.stdout)["format"]["duration"])


def extract_frames(video_path, output_dir, interval_sec, clear_existing=True):
    """
    Ekstrak frame per interval detik dari video.

    Args:
        video_path     : path ke file video
        output_dir     : folder untuk simpan frame
        interval_sec   : interval antar frame (detik)
        clear_existing : hapus frame lama sebelum ekstraksi baru

    Returns:
        list of Path: daftar file frame yang diekstrak
    """
    output_dir = Path(output_dir)

    if clear_existing and output_dir.exists():
        shutil.rmtree(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    # Durasi video
    duration = get_video_duration(video_path)

    # Titik sampling
    sampling_points = list(range(interval_sec, int(duration) + 1, interval_sec))
    if not sampling_points:
        sampling_points = [max(1, int(duration / 2))]

    # Ekstraksi per frame
    for t in tqdm(sampling_points, desc=f"  Frame [{Path(video_path).name[:30]}]", leave=False):
        out_file = output_dir / f"frame_{t:04d}s.jpg"
        subprocess.run(
            ["ffmpeg", "-y", "-loglevel", "error",
             "-ss", str(t), "-i", str(video_path),
             "-frames:v", "1", "-q:v", "2", str(out_file)],
            check=True
        )

    frames = sorted(output_dir.glob("frame_*.jpg"))
    return frames, duration


# === TEST: Ekstraksi untuk video pertama di daftar ===
if not selected_videos["paths"]:
    raise RuntimeError("Belum ada video dipilih. Kembali ke Cell 3.")

test_video_path = selected_videos["paths"][0]
test_video_name = Path(test_video_path).stem
safe_name = re.sub(r'[^\w\s-]', '', test_video_name).strip()
safe_name = re.sub(r'[-\s]+', '_', safe_name) or "video"

test_output_dir = Path(f"/content/output_{safe_name}/frames")

print(f"Test ekstraksi frame untuk video pertama:")
print(f"  Video  : {Path(test_video_path).name}")
print(f"  Output : {test_output_dir}\n")

test_frames, test_duration = extract_frames(
    test_video_path,
    test_output_dir,
    SAMPLING_INTERVAL,
    clear_existing=True
)

print(f"\nDurasi video : {test_duration:.2f} detik ({test_duration/60:.2f} menit)")
print(f"Frame terekstrak: {len(test_frames)}")
if test_frames:
    print(f"Frame pertama   : {test_frames[0].name}")
    print(f"Frame terakhir  : {test_frames[-1].name}")

Test ekstraksi frame untuk video pertama:
  Video  : Ini kata pendiri Zenius ＂Sabda PS＂ [PRLQSBl1v-I].mp4
  Output : /content/output_Ini_kata_pendiri_Zenius_Sabda_PS_PRLQSBl1v_I/frames




Durasi video : 55.98 detik (0.93 menit)
Frame terekstrak: 11
Frame pertama   : frame_0005s.jpg
Frame terakhir  : frame_0055s.jpg


In [ ]:
# ==============================================================
# CELL 5: TRANSKRIPSI WHISPER TURBO - FUNGSI REUSABLE
# Load model sekali, transkripsi banyak video
# ==============================================================

import whisper
import torch

# Load Whisper sekali di awal (akan dipakai untuk semua video)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Memuat Whisper {WHISPER_MODEL} ke {device}...")
whisper_model = whisper.load_model(WHISPER_MODEL, device=device)
print("Whisper siap.\n")


def transcribe_video(video_path, model):
    """
    Transkripsi video dengan word-level timestamp.

    Args:
        video_path : path ke file video
        model      : instance Whisper yang sudah di-load

    Returns:
        dict: {"language": str, "words": list of {start, end, word}}
    """
    result = model.transcribe(str(video_path), word_timestamps=True, verbose=False)

    all_words = []
    for seg in result["segments"]:
        for w in seg.get("words", []):
            all_words.append({
                "start": w["start"],
                "end": w["end"],
                "word": w["word"].strip()
            })

    return {
        "language": result.get("language", "unknown"),
        "words": all_words,
        "num_segments": len(result["segments"])
    }


def get_transcript_for_frame(timestamp_sec, words_list, window_sec):
    """Ambil transkrip dalam window [t - window, t]."""
    start = max(0, timestamp_sec - window_sec)
    end = timestamp_sec

    words = [w["word"] for w in words_list
             if w["start"] >= start and w["end"] <= end]

    if not words:
        words = [w["word"] for w in words_list
                 if w["start"] < end and w["end"] > start]

    return " ".join(words).strip() if words else "[no speech]"


# === TEST: Transkripsi video pertama ===
print(f"Test transkripsi untuk video pertama...")
print(f"  Video: {Path(test_video_path).name}\n")

test_transcript = transcribe_video(test_video_path, whisper_model)

print(f"Bahasa terdeteksi : {test_transcript['language']}")
print(f"Jumlah segmen     : {test_transcript['num_segments']}")
print(f"Kata dengan timestamp: {len(test_transcript['words'])}")

# Tes fungsi sinkronisasi di frame pertama
if test_frames:
    first_ts = int(test_frames[0].stem.replace("frame_", "").replace("s", ""))
    sample_text = get_transcript_for_frame(first_ts, test_transcript['words'], TRANSCRIPT_WINDOW)
    print(f"\nContoh transkrip di detik ke-{first_ts} (window {TRANSCRIPT_WINDOW}s ke belakang):")
    print(f"  {sample_text[:250]}")

# Simpan untuk dipakai di cell berikutnya
test_words = test_transcript['words']

Memuat Whisper turbo ke cuda...


100%|██████████████████████████████████████| 1.51G/1.51G [00:05<00:00, 286MiB/s]


Whisper siap.

Test transkripsi untuk video pertama...
  Video: Ini kata pendiri Zenius ＂Sabda PS＂ [PRLQSBl1v-I].mp4

Detected language: Indonesian


100%|██████████| 5598/5598 [00:16<00:00, 344.52frames/s]

Bahasa terdeteksi : id
Jumlah segmen     : 15
Kata dengan timestamp: 171

Contoh transkrip di detik ke-5 (window 5s ke belakang):
  Gue terakhir tes 160, tapi ada tapi -nya nih, gue tes pertama kali itu sebenernya 114.


In [ ]:
# ==============================================================
# CELL 6: LOAD QWEN2.5-VL 7B INSTRUCT
# Whisper DIBIARKAN di VRAM agar tidak perlu reload per video
# Download pertama: 5-8 menit (~15 GB), setelahnya cached
# ==============================================================

from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info

# Cek VRAM sebelum load VLM
vram_before = torch.cuda.memory_allocated() / (1024**3)
vram_total = torch.cuda.get_device_properties(0).total_memory / (1024**3)
print(f"VRAM sebelum load VLM: {vram_before:.2f} GB / {vram_total:.2f} GB")
print(f"Sisa VRAM            : {vram_total - vram_before:.2f} GB\n")

print(f"Memuat {VLM_MODEL_ID} ke L4 GPU...")
print("Proses ini butuh 5-8 menit pada download pertama, lalu cached.\n")

vlm_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    VLM_MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="sdpa"        # SDPA lebih cepat dari default
)
vlm_model.eval()

vlm_processor = AutoProcessor.from_pretrained(
    VLM_MODEL_ID,
    min_pixels=256*28*28,             # batas minimum image token
    max_pixels=1280*28*28             # batas maksimum (lebih kecil = lebih cepat)
)

# Report VRAM setelah load
vram_after = torch.cuda.memory_allocated() / (1024**3)
print(f"\nVLM siap.")
print(f"VRAM terpakai total : {vram_after:.2f} GB / {vram_total:.2f} GB")
print(f"Sisa VRAM           : {vram_total - vram_after:.2f} GB")

if vram_total - vram_after < 3:
    print("\nPERINGATAN: Sisa VRAM tipis. Mungkin perlu turunkan VLM_BATCH_SIZE ke 2.")
else:
    print(f"\nBatch size {VLM_BATCH_SIZE} aman untuk sisa VRAM ini.")

VRAM sebelum load VLM: 3.09 GB / 22.03 GB
Sisa VRAM            : 18.94 GB

Memuat Qwen/Qwen2.5-VL-7B-Instruct ke L4 GPU...
Proses ini butuh 5-8 menit pada download pertama, lalu cached.



config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]


VLM siap.
VRAM terpakai total : 18.55 GB / 22.03 GB
Sisa VRAM           : 3.49 GB

Batch size 4 aman untuk sisa VRAM ini.


In [ ]:
# ==============================================================
# CELL 6B: LEPAS WHISPER DARI VRAM
# Dibutuhkan karena GPU RAM terlalu tipis untuk batch inference VLM
# ==============================================================

import gc

# Simpan dulu hasil transkripsi test ke cache agar tidak hilang
transcripts_cache = {test_video_path: test_transcript}

# Cek VRAM sebelum
vram_before = torch.cuda.memory_allocated() / (1024**3)
print(f"VRAM sebelum lepas Whisper : {vram_before:.2f} GB")

# Lepas Whisper
try:
    del whisper_model
except NameError:
    print("(Whisper sudah tidak ada di scope, lanjut bersihkan cache)")

gc.collect()
torch.cuda.empty_cache()

# Cek VRAM

VRAM sebelum lepas Whisper : 18.55 GB


In [ ]:
# ==============================================================
# CELL 7: PROMPT TUNGGAL TERSTRUKTUR + BATCH INFERENCE
# Satu inferensi per frame menghasilkan 3 section (face, body, combined)
# Plus dukungan batch processing untuk speedup 2-3x tambahan
# ==============================================================

import re


# === PROMPT TUNGGAL TERSTRUKTUR ===
PROMPT_STRUCTURED = """You are an expert in nonverbal communication analysis.
Analyze the primary person in this image and provide three labeled sections.

Use formal academic English. Be concise: 2 sentences per section maximum.
If no face or person is visible, state that explicitly in the relevant section.

Output format (follow EXACTLY, do not add any extra text):

FACE: [facial expression: muscle movement, eye expression, mouth configuration, gaze, affective state]

BODY: [body language: hand position, gestures, arm configuration, shoulder orientation, torso posture]

COMBINED: [holistic interpretation integrating face and body; communicative intent from the combined signals]"""


# === FUNGSI INFERENSI SINGLE FRAME (BACKUP/DEBUGGING) ===
def run_vlm_single(image_path, prompt_text, max_new_tokens):
    """Inferensi satu gambar. Dipakai untuk testing/debugging."""
    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": str(image_path)},
            {"type": "text", "text": prompt_text}
        ]
    }]
    text = vlm_processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    imgs, vids = process_vision_info(messages)
    inputs = vlm_processor(text=[text], images=imgs, videos=vids,
                           padding=True, return_tensors="pt").to("cuda")
    with torch.no_grad():
        gen = vlm_model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    trimmed = [o[len(i):] for i, o in zip(inputs.input_ids, gen)]
    return vlm_processor.batch_decode(trimmed, skip_special_tokens=True,
                                       clean_up_tokenization_spaces=False)[0].strip()


# === FUNGSI INFERENSI BATCH (UTAMA, JAUH LEBIH CEPAT) ===
def run_vlm_batch(image_paths, prompt_text, max_new_tokens):
    """
    Inferensi beberapa gambar sekaligus dalam satu forward pass.
    Memanfaatkan parallelism GPU untuk speedup signifikan.

    Args:
        image_paths     : list of str/Path, daftar file gambar dalam batch
        prompt_text     : prompt yang sama diterapkan ke semua gambar
        max_new_tokens  : batas token yang di-generate per gambar

    Returns:
        list of str: output VLM untuk setiap gambar, urutan sama dengan input
    """
    # Bangun messages untuk setiap gambar
    batch_messages = []
    for img_path in image_paths:
        messages = [{
            "role": "user",
            "content": [
                {"type": "image", "image": str(img_path)},
                {"type": "text", "text": prompt_text}
            ]
        }]
        batch_messages.append(messages)

    # Apply chat template ke semua
    texts = [
        vlm_processor.apply_chat_template(m, tokenize=False, add_generation_prompt=True)
        for m in batch_messages
    ]

    # Process vision info untuk semua messages
    all_images = []
    for m in batch_messages:
        imgs, _ = process_vision_info(m)
        all_images.extend(imgs)

    # Tokenize batch
    inputs = vlm_processor(
        text=texts,
        images=all_images,
        videos=None,
        padding=True,
        return_tensors="pt"
    ).to("cuda")

    # Generate batch
    with torch.no_grad():
        gen = vlm_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=vlm_processor.tokenizer.pad_token_id
        )

    # Trim prompt dari output
    trimmed = [o[len(i):] for i, o in zip(inputs.input_ids, gen)]

    # Decode
    outputs = vlm_processor.batch_decode(
        trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )

    return [o.strip() for o in outputs]


# === PARSER OUTPUT TERSTRUKTUR ===
def parse_structured_output(raw_text):
    """
    Parse output VLM terstruktur menjadi tiga section: face, body, combined.
    Robust terhadap variasi formatting dari model.
    """
    face_match = re.search(r'FACE\s*:\s*(.*?)(?=\n\s*BODY\s*:|$)',
                           raw_text, re.DOTALL | re.IGNORECASE)
    body_match = re.search(r'BODY\s*:\s*(.*?)(?=\n\s*COMBINED\s*:|$)',
                           raw_text, re.DOTALL | re.IGNORECASE)
    combined_match = re.search(r'COMBINED\s*:\s*(.*?)$',
                               raw_text, re.DOTALL | re.IGNORECASE)

    face = face_match.group(1).strip() if face_match else "[parse failed: no FACE section]"
    body = body_match.group(1).strip() if body_match else "[parse failed: no BODY section]"
    combined = combined_match.group(1).strip() if combined_match else "[parse failed: no COMBINED section]"

    # Bersihkan sisa newline berlebih
    face = re.sub(r'\n+', ' ', face)
    body = re.sub(r'\n+', ' ', body)
    combined = re.sub(r'\n+', ' ', combined)

    return face, body, combined


# === TEST: Coba batch inference pada beberapa frame pertama ===
print("Test batch inference untuk validasi fungsi...\n")

test_batch = test_frames[:min(VLM_BATCH_SIZE, len(test_frames))]
print(f"Test batch ukuran {len(test_batch)} frame dari video: {Path(test_video_path).name}\n")

import time
start = time.time()
test_outputs = run_vlm_batch(test_batch, PROMPT_STRUCTURED, VLM_MAX_NEW_TOKENS)
elapsed = time.time() - start

print(f"Batch selesai dalam {elapsed:.1f} detik")
print(f"Rata-rata: {elapsed/len(test_batch):.1f} detik per frame\n")

# Parse dan tampilkan hasil test frame pertama
face, body, combined = parse_structured_output(test_outputs[0])
print(f"=== Contoh hasil parsing frame pertama ({test_batch[0].name}) ===")
print(f"\nFACE    : {face[:300]}")
print(f"\nBODY    : {body[:300]}")
print(f"\nCOMBINED: {combined[:300]}")
print(f"\n--- raw output length: {len(test_outputs[0])} chars ---")

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Test batch inference untuk validasi fungsi...

Test batch ukuran 4 frame dari video: Ini kata pendiri Zenius ＂Sabda PS＂ [PRLQSBl1v-I].mp4

Batch selesai dalam 11.1 detik
Rata-rata: 2.8 detik per frame

=== Contoh hasil parsing frame pertama (frame_0005s.jpg) ===

FACE    : The individual's facial expression appears neutral, with closed eyes and a slight smile, suggesting a relaxed or contemplative state. The mouth is slightly open, indicating a possible pause in speech or thought.

BODY    : The person's hands are positioned in front of them, with fingers slightly spread apart, which could indicate a gesture of explanation or emphasis. The arms are bent at the elbows, resting on what seems to be a table, suggesting a seated posture.

COMBINED: The combination of the neutral facial expression and the gesturing hands suggests that the individual might be engaged in a conversation or explaining something, possibly related to the context provided by the text overlay.

--- raw output length

In [ ]:
# ==============================================================
# CELL 8: FUNGSI ANALISIS VLM PER VIDEO DENGAN BATCH PROCESSING
# Dibungkus jadi fungsi reusable untuk batch loop di Cell 10
# ==============================================================

import time


def analyze_video_with_vlm(frame_files, batch_size, max_new_tokens, verbose=True):
    """
    Jalankan VLM structured prompt pada semua frame dari satu video.
    Menggunakan batch inference untuk efisiensi.

    Args:
        frame_files    : list of Path, daftar frame dari video
        batch_size     : jumlah frame per batch inference
        max_new_tokens : batas token output per frame
        verbose        : tampilkan progress

    Returns:
        list of dict: hasil per frame dengan keys:
            timestamp_sec, frame_file, face_interpretation,
            body_interpretation, combined_interpretation, raw_vlm_output
    """
    results = []
    total_frames = len(frame_files)
    num_batches = (total_frames + batch_size - 1) // batch_size

    if verbose:
        print(f"  VLM analysis: {total_frames} frame, {num_batches} batch (size {batch_size})")

    start_time = time.time()
    pbar = tqdm(total=total_frames, desc="  VLM", leave=False) if verbose else None

    for batch_idx in range(num_batches):
        batch_start = batch_idx * batch_size
        batch_end = min(batch_start + batch_size, total_frames)
        batch_frames = frame_files[batch_start:batch_end]

        # Coba batch inference dulu
        try:
            batch_outputs = run_vlm_batch(batch_frames, PROMPT_STRUCTURED, max_new_tokens)
        except torch.cuda.OutOfMemoryError:
            # Fallback: single inference satu per satu jika OOM
            if verbose:
                print(f"\n  [OOM pada batch {batch_idx+1}, fallback ke single inference]")
            torch.cuda.empty_cache()
            batch_outputs = []
            for f in batch_frames:
                try:
                    out = run_vlm_single(f, PROMPT_STRUCTURED, max_new_tokens)
                except Exception as e:
                    out = f"[Error single: {str(e)[:80]}]"
                batch_outputs.append(out)
        except Exception as e:
            # Fallback umum
            if verbose:
                print(f"\n  [Error batch: {str(e)[:80]}, fallback ke single]")
            batch_outputs = []
            for f in batch_frames:
                try:
                    out = run_vlm_single(f, PROMPT_STRUCTURED, max_new_tokens)
                except Exception as e2:
                    out = f"[Error: {str(e2)[:80]}]"
                batch_outputs.append(out)

        # Parse dan simpan hasil setiap frame di batch
        for frame_path, raw_output in zip(batch_frames, batch_outputs):
            timestamp = int(frame_path.stem.replace("frame_", "").replace("s", ""))
            face, body, combined = parse_structured_output(raw_output)

            results.append({
                "timestamp_sec": timestamp,
                "frame_file": frame_path.name,
                "face_interpretation": face,
                "body_interpretation": body,
                "combined_interpretation": combined,
                "raw_vlm_output": raw_output
            })

            if pbar:
                pbar.update(1)

    if pbar:
        pbar.close()

    elapsed = time.time() - start_time
    if verbose:
        print(f"  Selesai: {elapsed:.1f}s total, {elapsed/total_frames:.1f}s per frame")

    return results, elapsed


# === TEST: Analisis lengkap video pertama ===
print(f"Test analisis VLM lengkap untuk video pertama:\n")

test_vlm_results, test_elapsed = analyze_video_with_vlm(
    test_frames,
    batch_size=VLM_BATCH_SIZE,
    max_new_tokens=VLM_MAX_NEW_TOKENS,
    verbose=True
)

# Tampilkan ringkasan
print(f"\n{'=' * 60}")
print(f"RINGKASAN TEST")
print(f"{'=' * 60}")
print(f"Total frame dianalisis : {len(test_vlm_results)}")
print(f"Total waktu           : {test_elapsed:.1f} detik ({test_elapsed/60:.2f} menit)")
print(f"Rata-rata per frame   : {test_elapsed/len(test_vlm_results):.1f} detik")
print(f"Proyeksi untuk 1 video 5 menit (60 frame): {test_elapsed/len(test_vlm_results) * 60 / 60:.1f} menit")
print(f"Proyeksi untuk 100 video tersebut         : {test_elapsed/len(test_vlm_results) * 60 * 100 / 3600:.1f} jam")

# Contoh output 3 frame pertama
print(f"\n{'=' * 60}")
print(f"CONTOH OUTPUT 3 FRAME PERTAMA")
print(f"{'=' * 60}")
for r in test_vlm_results[:3]:
    print(f"\n--- Frame {r['frame_file']} (detik ke-{r['timestamp_sec']}) ---")
    print(f"FACE    : {r['face_interpretation'][:200]}")
    print(f"BODY    : {r['body_interpretation'][:200]}")
    print(f"COMBINED: {r['combined_interpretation'][:200]}")

Test analisis VLM lengkap untuk video pertama:

  VLM analysis: 11 frame, 3 batch (size 4)


  Selesai: 33.1s total, 3.0s per frame

RINGKASAN TEST
Total frame dianalisis : 11
Total waktu           : 33.1 detik (0.55 menit)
Rata-rata per frame   : 3.0 detik
Proyeksi untuk 1 video 5 menit (60 frame): 3.0 menit
Proyeksi untuk 100 video tersebut         : 5.0 jam

CONTOH OUTPUT 3 FRAME PERTAMA

--- Frame frame_0005s.jpg (detik ke-5) ---
FACE    : The individual's facial expression appears neutral, with closed eyes and a slight smile, suggesting a relaxed or contemplative state. The mouth is slightly open, indicating a possible pause in speech 
BODY    : The person's hands are positioned in front of them, with fingers slightly spread apart, which could indicate a gesture of explanation or emphasis. The arms are bent at the elbows, resting on what seem
COMBINED: The combination of the neutral facial expression and the gesturing hands suggests that the individual might be engaged in a conversation or explaining something, possibly related to the context provid

--- Frame frame_0010s

In [ ]:
# ==============================================================
# CELL 9 (REVISI): FIX METADATA REMOVAL
# Perbaikan error openpyxl saat set properties timestamp ke None
# ==============================================================

from openpyxl import Workbook
from openpyxl.drawing.image import Image as XLImage
from openpyxl.styles import Font, Alignment, PatternFill
from openpyxl.utils import get_column_letter
from PIL import Image as PILImage
import pandas as pd


def strip_excel_metadata(wb):
    """
    Hapus/kosongkan metadata sensitif dari workbook openpyxl.
    Field timestamp tidak di-set None (menyebabkan error), cukup dibiarkan default.
    """
    try:
        wb.properties.creator = ""
        wb.properties.lastModifiedBy = ""
        wb.properties.title = ""
        wb.properties.subject = ""
        wb.properties.description = ""
        wb.properties.keywords = ""
        wb.properties.category = ""
        wb.properties.identifier = ""
        wb.properties.language = ""
        wb.properties.revision = ""
    except Exception as e:
        print(f"  (warning saat strip metadata: {str(e)[:80]})")
    return wb


def export_to_excel(video_name, vlm_results, frames_dir, words_list,
                    transcript_window, excel_path, csv_path=None):
    """
    Konsolidasi hasil VLM + transkrip, export ke Excel dengan gambar embedded.
    """
    # Konsolidasi data
    final_rows = []
    for v in vlm_results:
        ts = v["timestamp_sec"]
        transcript = get_transcript_for_frame(ts, words_list, transcript_window)

        final_rows.append({
            "timestamp_sec": ts,
            "timestamp_mmss": f"{ts//60:02d}:{ts%60:02d}",
            "frame_file": v["frame_file"],
            "frame_path": str(Path(frames_dir) / v["frame_file"]),
            "transcript": transcript,
            "face": v["face_interpretation"],
            "body": v["body_interpretation"],
            "combined": v["combined_interpretation"],
            "raw_vlm": v.get("raw_vlm_output", "")
        })

    # Build Excel workbook
    wb = Workbook()
    ws = wb.active
    ws.title = "Multimodal Analysis"[:31]

    headers = [
        "Timestamp (sec)", "Timestamp (mm:ss)", "Frame Image",
        "Transcript", "Face Interpretation", "Body Interpretation",
        "Combined Interpretation", "Raw VLM Output"
    ]

    header_fill = PatternFill(start_color="1F4E78", end_color="1F4E78", fill_type="solid")
    header_font = Font(bold=True, color="FFFFFF", size=11)

    for col, h in enumerate(headers, 1):
        c = ws.cell(row=1, column=col, value=h)
        c.fill = header_fill
        c.font = header_font
        c.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)

    widths = {1: 12, 2: 14, 3: 22, 4: 42, 5: 52, 6: 52, 7: 62, 8: 45}
    for col, w in widths.items():
        ws.column_dimensions[get_column_letter(col)].width = w

    for idx, r in enumerate(final_rows, start=2):
        ws.row_dimensions[idx].height = 120

        ws.cell(row=idx, column=1, value=r["timestamp_sec"])
        ws.cell(row=idx, column=2, value=r["timestamp_mmss"])
        for col, key in [(4, "transcript"), (5, "face"),
                         (6, "body"), (7, "combined"), (8, "raw_vlm")]:
            cell = ws.cell(row=idx, column=col, value=r[key])
            cell.alignment = Alignment(wrap_text=True, vertical="top")

        try:
            pil = PILImage.open(r["frame_path"])
            pil.thumbnail((150, 150))
            thumb = f"{r['frame_path']}.thumb.png"
            pil.save(thumb)
            xl_img = XLImage(thumb)
            xl_img.anchor = f"C{idx}"
            ws.add_image(xl_img)
        except Exception as e:
            ws.cell(row=idx, column=3, value=f"[img err: {str(e)[:40]}]")

    # Strip metadata (versi aman yang tidak error)
    strip_excel_metadata(wb)

    # Simpan
    Path(excel_path).parent.mkdir(parents=True, exist_ok=True)
    wb.save(str(excel_path))

    # CSV backup
    if csv_path:
        Path(csv_path).parent.mkdir(parents=True, exist_ok=True)
        pd.DataFrame(final_rows).to_csv(str(csv_path), index=False)

    return excel_path, csv_path, len(final_rows)


# === TEST: Export Excel untuk video test ===
print(f"Test export Excel untuk video pertama...\n")

test_safe_name = re.sub(r'[^\w\s-]', '', Path(test_video_path).stem).strip()
test_safe_name = re.sub(r'[-\s]+', '_', test_safe_name) or "video"
test_excel_path = Path(f"/content/output_{test_safe_name}/analysis_{test_safe_name}.xlsx")
test_csv_path = Path(f"/content/output_{test_safe_name}/analysis_{test_safe_name}.csv")
test_frames_dir = test_excel_path.parent / "frames"

excel_out, csv_out, n_rows = export_to_excel(
    video_name=test_safe_name,
    vlm_results=test_vlm_results,
    frames_dir=test_frames_dir,
    words_list=test_words,
    transcript_window=TRANSCRIPT_WINDOW,
    excel_path=test_excel_path,
    csv_path=test_csv_path
)

print(f"Excel tersimpan : {excel_out}")
print(f"CSV tersimpan   : {csv_out}")
print(f"Jumlah baris    : {n_rows}")

excel_size_kb = excel_out.stat().st_size / 1024
csv_size_kb = csv_out.stat().st_size / 1024 if csv_out.exists() else 0
print(f"\nUkuran Excel : {excel_size_kb:.1f} KB")
print(f"Ukuran CSV   : {csv_size_kb:.1f} KB")

Test export Excel untuk video pertama...

Excel tersimpan : /content/output_Ini_kata_pendiri_Zenius_Sabda_PS_PRLQSBl1v_I/analysis_Ini_kata_pendiri_Zenius_Sabda_PS_PRLQSBl1v_I.xlsx
CSV tersimpan   : /content/output_Ini_kata_pendiri_Zenius_Sabda_PS_PRLQSBl1v_I/analysis_Ini_kata_pendiri_Zenius_Sabda_PS_PRLQSBl1v_I.csv
Jumlah baris    : 11

Ukuran Excel : 150.0 KB
Ukuran CSV   : 18.0 KB
